# H3L3 — Lookups, running states and chain-of-thought

**Notebook 5 of this week.** H3L2 showed that attention replaced recurrence
because it reaches any position in one step. This notebook asks where that
helps and where it does not. The parts follow the slides:

1. how far a stack of layers can see: two attention layers build a trigram
2. lookups, where attention wins: recalling the value stored under a key
3. a running state, where a recurrent network wins: the parity of a bit string
4. chain-of-thought: the transformer writes the running state and recovers

**Colab:** Runtime → Run all. About two minutes on the CPU runtime.


In [ ]:
import math, os, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)
torch.set_num_threads(2)                       # as on the free Colab runtime
C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"
FIGDIR = os.environ.get("SEMINAR_FIGDIR")


def save(fig, name):
    """Only when the slides are built: store the figure for the deck."""
    if FIGDIR:
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"), bbox_inches="tight", dpi=200)


## Part 1 — How far a stack of layers can see

We build a two-layer attention network by hand, with no training. The words
are *a*, *b* and *c*, each a one-hot vector of length 3. The vector of each
token has three slots of length 3: slot **A** holds the current word, slot
**B** the previous word and slot **C** the word two back. A first token
`BOS` (with an all-zero word) marks the start.

Each layer has one **previous-token head**: its queries and keys are computed
from the positions so that position $i$ puts all its weight on position
$i - 1$. The head of layer 1 copies slot A of the previous token into slot
B; the head of layer 2 copies slot B of the previous token into slot C.


In [ ]:
WORDS = ["a", "b", "c"]
text = ["BOS", "a", "b", "c"]
T, V = len(text), len(WORDS)
A, B, C = slice(0, 3), slice(3, 6), slice(6, 9)

X = np.zeros((T, 9))                           # slot A: the current word
for i, w in enumerate(text):
    if w in WORDS:
        X[i, WORDS.index(w)] = 1.0


def previous_token_weights(T):
    """Scores from positions: 10 for j = i - 1, 0 otherwise; causal softmax."""
    S = np.where(np.arange(T)[None, :] == np.arange(T)[:, None] - 1, 10.0, 0.0)
    S[np.triu_indices(T, 1)] = -np.inf
    P = np.exp(S - S.max(1, keepdims=True))
    return P / P.sum(1, keepdims=True)


P = previous_token_weights(T)
# layer 1: the previous word -> slot B
X1 = X.copy(); X1[:, B] += P @ X[:, A]
# layer 2: its previous word -> slot C
X2 = X1.copy(); X2[:, C] += P @ X1[:, B]

fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.6))
for ax, M, title in zip(axes, [X, X1, X2], ["after the embedding", "after layer 1", "after layer 2"]):
    ax.imshow(M.round(2), cmap="Blues", vmin=0, vmax=1)
    ax.set_yticks(range(T), text); ax.set_xticks([1, 4, 7], ["A: word", "B: previous", "C: two back"])
    for x in (2.5, 5.5):
        ax.axvline(x, color=C_GRAY, lw=0.8)
    ax.set_title(title, fontsize=10)
plt.tight_layout(); save(fig, "fig-t-w5-h3l3-p1"); plt.show()

show = lambda v: [WORDS[k] if v[k] > 0.5 else "" for k in range(3)]
print("position of c after layer 2:", "A", show(X2[3, A]), "B", show(X2[3, B]), "C", show(X2[3, C]))


Each layer extends what a position holds by one token, for all positions at
once. With heads that look back a fixed number of positions, $L$ layers see
the current token and $L$ tokens before it, like a recurrent network unrolled
over $L$ steps. Attention is not limited to fixed offsets, however: a head can
also find a position by its content, at any distance, as the next part shows.


**Exercise.** With previous-token heads only, how many layers does a position need to hold the word four positions back?

<details><summary>Answer</summary>

Four, one layer per step back. A head that finds the word by its content, as in Part 2, needs only one.

</details>


## Part 2 — Lookups, where attention wins

Each example lists $k$ pairs of a key and a value, then a query key; the
model must answer with the value stored under that key:

```
k5 v12  k2 v7  k9 v3  ...  ?  k2   ->  v7
```

The answer is somewhere in the input and can be found by matching the query
with the keys: a lookup. We train a recurrent network (GRU) and a two-layer
transformer of the same width on 2 to 16 pairs.

Both transformers in this notebook get their positions from a fixed penalty
on the scores that grows with the distance (ALiBi), with a different penalty
per head and one head without any. Unlike learned position vectors, this
works at lengths never seen in training, so that Parts 3 and 4 test the
computation, not the positions.


In [ ]:
D, BATCH = 64, 128


class Block(nn.Module):
    def __init__(self, H=4, use_mlp=True):
        super().__init__()
        self.use_mlp = use_mlp
        self.H, self.dh = H, D // H
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.qkv = nn.Linear(D, 3 * D, bias=False)
        self.out = nn.Linear(D, D, bias=False)
        self.mlp = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))
        self.slopes = torch.tensor([1.0, 0.25, 1 / 16, 0.0]).view(1, H, 1, 1)

    def forward(self, x):
        Bn, Tn, _ = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(Bn, Tn, 3, self.H, self.dh).permute(2, 0, 3, 1, 4)
        i = torch.arange(Tn)
        dist = (i[:, None] - i[None, :]).float()          # how far back key j is
        s = q @ k.transpose(-1, -2) / math.sqrt(self.dh) - self.slopes * dist.clamp(min=0)
        s = s.masked_fill(dist < 0, -torch.inf)           # the causal mask
        x = x + self.out((F.softmax(s, -1) @ v).transpose(1, 2).reshape(Bn, Tn, D))
        return x + self.mlp(self.ln2(x)) if self.use_mlp else x


class Transformer(nn.Module):
    def __init__(self, vocab, n_layers=2, use_mlp=True):
        super().__init__()
        self.emb = nn.Embedding(vocab, D)
        self.blocks = nn.ModuleList(Block(use_mlp=use_mlp) for _ in range(n_layers))
        self.head = nn.Linear(D, vocab)

    def forward(self, x):
        h = self.emb(x)
        for b in self.blocks:
            h = b(h)
        return self.head(h)


class Recurrent(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.emb = nn.Embedding(vocab, D)
        self.rnn = nn.GRU(D, D, batch_first=True)
        self.head = nn.Linear(D, vocab)

    def forward(self, x):
        return self.head(self.rnn(self.emb(x))[0])


def train(model, make, steps):
    torch.manual_seed(1)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    for _ in range(steps):
        x, y, m = make(BATCH)
        loss = F.cross_entropy(model(x)[m], y[m])
        opt.zero_grad(); loss.backward(); opt.step()
    return model


NK, KEY0, VAL0, QUERY, V_REC = 16, 4, 20, 3, 36   # 16 keys, 16 values, a query token


def make_recall(n, k=None):
    k = k or int(torch.randint(2, 17, (1,)))
    keys = torch.stack([torch.randperm(NK)[:k] for _ in range(n)]) + KEY0
    vals = torch.randint(0, NK, (n, k)) + VAL0
    pick = torch.randint(0, k, (n,))
    x = torch.stack([keys, vals], 2).view(n, 2 * k)
    x = torch.cat([x, torch.full((n, 1), QUERY), keys[torch.arange(n), pick][:, None]], 1)
    y = torch.zeros_like(x); y[:, -1] = vals[torch.arange(n), pick]
    m = torch.zeros_like(x, dtype=torch.bool); m[:, -1] = True   # only the answer is scored
    return x, y, m


def recall_accuracy(model, k, n=512):
    with torch.no_grad():
        x, y, _ = make_recall(n, k)
        return (model(x)[:, -1].argmax(-1) == y[:, -1]).float().mean().item()


t0 = time.time()
rnn_rec = train(Recurrent(V_REC),
                make_recall, steps=2000)
tf_rec = train(Transformer(V_REC),
               make_recall, steps=2000)
print(f"trained in {time.time() - t0:.0f} s")

ks = [2, 4, 8, 12, 16]
acc_rnn = [recall_accuracy(rnn_rec, k) for k in ks]
acc_tf = [recall_accuracy(tf_rec, k) for k in ks]
for k, a, b in zip(ks, acc_rnn, acc_tf):
    print(f"k = {k:2d}   recurrent {a:.2f}   transformer {b:.2f}")

fig, ax = plt.subplots(figsize=(5.4, 3.3))
ax.plot(ks, acc_tf, "^-", color=C_BLUE, label="transformer")
ax.plot(ks, acc_rnn, "s-", color=C_TEAL, label="recurrent (GRU)")
ax.axhline(1 / NK, color=C_GRAY, ls=":", lw=1)
ax.set_xlabel("number of key-value pairs k"); ax.set_ylabel("accuracy")
ax.set_ylim(-0.04, 1.06); ax.legend(frameon=False)
ax.set_title("recalling the value of a key", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l3-p2"); plt.show()


The transformer finds the key by matching it, at any distance and for any
number of pairs. This is a lookup in the **context**: the answer is a value
computed from a token of this input. H1L1, Part 6 showed the other kind of
lookup, in the **weights**: the MLP supplied "Paris", which no token
contained. The recurrent network must carry all pairs in one state of
fixed size while reading them, and it loses them as $k$ grows.


**Exercise.** Is the lookup done by the attention or by the MLPs? Train `Transformer(V_REC, use_mlp=False)`, which has no MLPs, and measure its accuracy.

<details><summary>Answer</summary>

By the attention: without MLPs the accuracy is still 1.00 at every $k$. In layer 1, a head at each value looks one position back and copies its key into the value's vector; in layer 2, the query key matches that key and the head copies the value to the answer, the two steps of an induction head.

</details>


**Exercise.** Why does the recurrent network do best at $k = 2$?

<details><summary>Answer</summary>

Its state has to hold every pair it has read until the query arrives. Two pairs fit in the state of width 64; sixteen pairs, each one of 16 keys with one of 16 values, do not, and the older pairs are overwritten.

</details>


## Part 3 — A running state, where a recurrent network wins

The **parity** of a string of bits says whether it contains an odd number of
1s. A recurrent network needs to keep one bit of state and flip it at every
1, reading the bits in order. For attention the answer is not stored at any
position, so there is nothing to look up; averaging over the bits gives the
fraction of 1s, but parity depends on the exact count, and changing any
single bit changes the answer.

Both models are trained on strings of 1 to 16 bits to output, at every
position, the parity of the bits so far. We test the parity of whole strings
of up to 64 bits.


In [ ]:
TRAIN_MAX = 16


def make_parity(n, L=None):
    L = L or int(torch.randint(1, TRAIN_MAX + 1, (1,)))
    x = torch.randint(0, 2, (n, L))
    y = x.cumsum(1) % 2                                  # the running parity
    return x, y, torch.ones_like(x, dtype=torch.bool)


def parity_accuracy(model, L, n=512):
    with torch.no_grad():
        x = torch.randint(0, 2, (n, L))
        return (model(x)[:, -1].argmax(-1) == x.sum(1) % 2).float().mean().item()


t0 = time.time()
rnn_par = train(Recurrent(2),
                make_parity, steps=1500)
tf_par = train(Transformer(2),
               make_parity, steps=1500)
print(f"trained in {time.time() - t0:.0f} s")

Ls = [4, 8, 16, 32, 64]
par_rnn = [parity_accuracy(rnn_par, L) for L in Ls]
par_tf = [parity_accuracy(tf_par, L) for L in Ls]
for L, a, b in zip(Ls, par_rnn, par_tf):
    print(f"L = {L:2d}   recurrent {a:.2f}   transformer {b:.2f}")

fig, ax = plt.subplots(figsize=(5.4, 3.3))
ax.axvspan(1, TRAIN_MAX, color=C_GRAY, alpha=0.08, lw=0)
ax.text(TRAIN_MAX / 2, 0.08, "trained", ha="center", fontsize=8, color=C_GRAY)
ax.plot(Ls, par_rnn, "s-", color=C_TEAL, label="recurrent (GRU)")
ax.plot(Ls, par_tf, "^-", color=C_BLUE, label="transformer")
ax.axhline(0.5, color=C_GRAY, ls=":", lw=1)
ax.set_xscale("log", base=2); ax.minorticks_off(); ax.set_xticks(Ls, [str(L) for L in Ls])
ax.set_xlabel("length of the bit string"); ax.set_ylabel("accuracy of the parity")
ax.set_ylim(-0.04, 1.06); ax.legend(frameon=False, loc="lower left")
ax.set_title("the parity of a bit string", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l3-p3"); plt.show()


The recurrent network learns the flip and applies it at every length. The
transformer, computing each answer in one pass through two layers, is near
chance except on the shortest strings, even at lengths it was trained on.


**Exercise.** Why is the transformer still fairly accurate on strings of 4 bits?

<details><summary>Answer</summary>

There are only 16 strings of 4 bits, and the last position can attend to all of them at once; the model can learn the answer for each pattern of a few bits. The number of patterns doubles with every bit, and no fixed combination of two layers covers them all.

</details>


## Part 4 — Chain-of-thought: the running state as text

We let the transformer **write** the running parity. The sequence alternates
a bit and the parity so far, and the model writes each parity token itself:

```
bits:     1        0        1        1
written:     odd      odd      even     odd
sequence: 1  odd  0  odd  1  even  1  odd
```

To write a parity token, the model needs only the bit just before it and the
parity token it wrote before that, two positions back: two lookups and a flip.
The generation loop supplies the sequential steps; the written tokens carry
the state, as the hidden state does in a recurrent network. At test time the
model generates the parity tokens one at a time, from its own earlier output.


In [ ]:
EVEN, ODD = 2, 3                                          # tokens: bits 0/1, parity even/odd


def make_cot(n, L=None):
    L = L or int(torch.randint(1, TRAIN_MAX + 1, (1,)))
    bits = torch.randint(0, 2, (n, L))
    state = bits.cumsum(1) % 2 + EVEN
    seq = torch.stack([bits, state], 2).view(n, 2 * L)   # b1 s1 b2 s2 ...
    x, y = seq[:, :-1], seq[:, 1:]
    m = torch.zeros_like(y, dtype=torch.bool); m[:, 0::2] = True   # score the written states
    return x, y, m


def cot_accuracy(model, L, n=512):
    """Generate the parity tokens one at a time, from the model's own output."""
    with torch.no_grad():
        bits = torch.randint(0, 2, (n, L))
        seq = bits[:, :1]
        for t in range(L):
            out = model(seq)[:, -1]
            st = out[:, EVEN:].argmax(-1) + EVEN
            seq = torch.cat([seq, st[:, None]], 1)
            if t + 1 < L:
                nxt = bits[:, t + 1:t + 2]
                seq = torch.cat([seq, nxt], 1)
        return ((seq[:, -1] - EVEN) == bits.sum(1) % 2).float().mean().item()


t0 = time.time()
tf_cot = train(Transformer(4), make_cot, steps=1500)
print(f"trained in {time.time() - t0:.0f} s")
par_cot = [cot_accuracy(tf_cot, L) for L in Ls]
for L, a in zip(Ls, par_cot):
    print(f"L = {L:2d}   transformer with chain-of-thought {a:.2f}")

fig, ax = plt.subplots(figsize=(5.4, 3.3))
ax.axvspan(1, TRAIN_MAX, color=C_GRAY, alpha=0.08, lw=0)
ax.text(TRAIN_MAX / 2, 0.08, "trained", ha="center", fontsize=8, color=C_GRAY)
ax.plot(Ls, par_rnn, "s-", color=C_TEAL, label="recurrent (GRU)")
ax.plot(Ls, par_tf, "^-", color=C_BLUE, label="transformer, one pass")
ax.plot(Ls, par_cot, "o--", color=C_RED, label="transformer, chain-of-thought")
ax.axhline(0.5, color=C_GRAY, ls=":", lw=1)
ax.set_xscale("log", base=2); ax.minorticks_off(); ax.set_xticks(Ls, [str(L) for L in Ls])
ax.set_xlabel("length of the bit string"); ax.set_ylabel("accuracy of the parity")
ax.set_ylim(-0.04, 1.06); ax.legend(frameon=False, loc="lower left", fontsize=8)
ax.set_title("the parity, with the running state written out", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l3-p4"); plt.show()


With the running state written out, the same kind of transformer computes
the parity at every length, including four times the longest training string.


**Exercise.** In the chain-of-thought sequence, at which positions are the two tokens the model needs to write the next parity token?

<details><summary>Answer</summary>

The bit is the token just before it (one position back), and the previous parity token is two positions back. Both are at fixed, short distances, which a head with a steep distance penalty finds easily, at any length.

</details>


## What this notebook showed

| task | recurrent network | transformer, one pass | transformer, chain-of-thought |
|---|---|---|---|
| recall a value by its key (a lookup) | falls as the number of pairs grows | correct at every number of pairs | — |
| parity (a running state) | correct at every length | near chance beyond the shortest strings | correct at every length |

Attention is strong when the answer is somewhere in the input and can be
found; a recurrent network is strong when the answer must be built up step
by step. Writing the intermediate steps as text turns a running state back
into a sequence of lookups.
